# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 code cell
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_avg_position, scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

def reason_code(row):
    if row['gsc_impressions'] == 0:
        return 'NO_VISIBILITY'
    elif row['gsc_avg_position'] <= 10:
        return 'RANKS_WELL'
    else:
        return 'VISIBLE_BUT_POOR_RANK'

df['reason_code'] = df.apply(reason_code, axis=1)
df['score'] = np.where(
    df['reason_code'] == 'VISIBLE_BUT_POOR_RANK',
    df['gsc_impressions'] * (df['gsc_avg_position'] - 10),
    0
)
threshold = df.loc[df['score'] > 0, 'score'].quantile(0.95)
df['action'] = np.where(df['score'] >= threshold, 'REVIEW_FOR_RANKING_FIX', 'MONITOR')

# Aggregate to one row per content item, not per row-day (the Week 4 fix identified in the pressure test)
queue = (df[df['action']=='REVIEW_FOR_RANKING_FIX']
    .groupby(['client_hash_id','content_hash_id'])
    .agg(avg_score=('score','mean'), days_flagged=('score','count'),
         avg_impressions=('gsc_impressions','mean'), avg_position=('gsc_avg_position','mean'))
    .reset_index()
    .sort_values('avg_score', ascending=False))

print("Unique content items flagged:", len(queue))
print(queue.head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique content items flagged: 6036
               client_hash_id           content_hash_id      avg_score  \
1830  client_23a62021009f63c4  content_36e53e9c707674fc  143028.000000   
2458  client_23a62021009f63c4  content_6530fa9d297c46eb  138552.000000   
2663  client_23a62021009f63c4  content_73aa61dcedebbf30   94038.258065   
2678  client_23a62021009f63c4  content_74de5f247659e956   93498.000000   
2237  client_23a62021009f63c4  content_559cdd76da9306de   84493.870968   
3452  client_23a62021009f63c4  content_ab91e088440ace78   83755.451613   
4986  client_e547b89c05043229  content_175ec61005922428   82465.000000   
2278  client_23a62021009f63c4  content_580e7d0863f1bc6d   76439.333333   
3707  client_23a62021009f63c4  content_bdf60c86117079be   75368.354839   
5408  client_e5c2aa26a8598242  content_3acc45f5ce928b32   75266.000000   

      days_flagged  avg_impressions  avg_position  
1830            31      6276.741935     32.766674  
2458             5      1759.400000     87.288

The ranked queue flagged 6,036 unique content items as VISIBLE_BUT_POOR_RANK, aggregated from content-day rows to one row per content item (avg score, days flagged, avg impressions, avg position). The top-ranked item (content_36e53e9c..., avg score 143,028) was flagged all 31 days of March — a stable, persistent pattern, not a fluke. By contrast, some lower-ranked items (e.g., avg score ~75,000) were flagged only 1 day out of 31 — a single-day spike, not a sustained issue. This distinction matters for prioritization: a human reviewer should weight days_flagged alongside avg_score, since a high score from one unusual day is weaker evidence than a moderate score sustained across the whole month.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Intended use: A content team lead or SEO strategist uses this queue as a starting shortlist for manual review — which pages might benefit from a ranking-focused fix (title/meta, internal linking, content depth) — not as an automated action list.

Limits, stated plainly: This is a rule-based ranking, not a validated predictive model. The Week 5/6 regression model (Random Forest, grouped-by-client split) scored R² = -0.0074 — worse than predicting the client average — meaning it does not reliably predict gsc_avg_position for a new client from same-day behavioral signals alone. That model is explicitly not used in this playbook. The rule-based score only reflects observed volume × position gap in March 2026; it says nothing about why a page ranks poorly (content quality, technical SEO, backlinks — none of which are in this dataset), and nothing about whether a fix will actually work.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


A human must check, before acting on any flagged item:

Whether the poor position reflects a genuine on-page/technical issue, or an irrelevant query match (a page can rank poorly for a query it was never meant to target)
Whether the page's topic is still commercially/strategically relevant to the client today
Whether flagged content-days for the same content item (see days_flagged) represent a stable, real pattern or a short-lived blip

What must NOT be automated:

No automatic content edits, republishing, or metadata changes based on this score alone
No automatic client-facing reporting claiming a page "will" improve if fixed — the data supports association, not a causal guarantee
No use of this queue to justify headcount, budget, or contract decisions without a human strategist's sign-off first.

Before acting on any flagged item, check days_flagged: items flagged fewer than ~5 days out of the month should be treated as low-confidence and deprioritized relative to items flagged across most or all of the month, even if their raw avg_score looks similar. A single-day flag is more likely to reflect a temporary ranking fluctuation than a real, fixable problem.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Since no ML model is in production here, "retrain" doesn't apply — the trigger question is when the rule itself needs re-validation, not when to retrain a model:

Re-run the Week 4 signal check (CTR vs. position) quarterly — if it stops showing a CONFIRMED monotonic pattern, the rule's core assumption no longer holds and the playbook should pause
If Google's SERP layout or ranking factors visibly shift (public knowledge, not in this dataset), re-validate before trusting the queue
If gsc_data_available coverage drops significantly month over month, the queue's reliability shrinks correspondingly and should be flagged in any handoff

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5 code cell
import os
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

queue.to_csv('work/outputs/content_action_playbook.csv', index=False)

import json
metrics = {
    "rule": "VISIBLE_BUT_POOR_RANK",
    "signal_1_ctr_vs_position": "CONFIRMED",
    "unique_content_items_flagged": len(queue),
    "week5_6_model_r2_grouped_split": -0.0074,
    "week5_6_model_used_in_playbook": False,
    "reason": "Model did not beat grouped-split baseline; playbook uses validated rule-based signal instead"
}
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported queue:", len(queue), "rows")
print("Exported metrics JSON")


Exported queue: 6036 rows
Exported metrics JSON


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.